# 모델 학습용 필요 환경 정리
팀원 공유용

In [ ]:
!nvidia-smi           # GPU 확인
!python --version
!pip install --upgrade pip


## 1. MediaPipe Pose (+Hands/Holistic) + TCN

### 필요 라이브러리
*   mediapipe

*   opencv-python

*   torch (TCN 구현용)

*   (선택) pytorch-lightning or torchmetrics 등

In [ ]:
!pip install --upgrade pip

# MediaPipe + 기본 유틸
!pip install mediapipe opencv-python

# Torch (Colab은 보통 torch 이미 설치되어 있으므로 버전 확인 후 필요 시 맞추기)
import torch, torchvision, torchaudio
print(torch.__version__)

# 그래도 명시적으로 깔고 싶으면 (CUDA 12.x용 최신 stable)
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### 간단 TCN 구조 예

In [ ]:
import torch
import torch.nn as nn

class SimpleTCN(nn.Module):
    def __init__(self, in_dim, hid_dim=128, num_classes=7, num_layers=3):
        super().__init__()
        layers = []
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                nn.Conv1d(in_dim, hid_dim, kernel_size=3, padding=dilation, dilation=dilation)
            )
            layers.append(nn.ReLU())
            in_dim = hid_dim
            dilation *= 2
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(hid_dim, num_classes)

    def forward(self, x):
        # x: (B, T, C) -> Conv1d는 (B, C, T)
        x = x.transpose(1, 2)
        h = self.tcn(x)              # (B, H, T)
        h = h.transpose(1, 2)        # (B, T, H)
        logits = self.head(h)        # (B, T, num_classes)
        return logits


## 2. YOLOv8n-Pose + TCN

### 필요 라이브러리
* ultralytics (YOLOv8)

* opencv-python

* torch, torchvision

In [ ]:
!pip install --upgrade pip

# YOLOv8 (ultralytics)
!pip install ultralytics

# Torch (필요시 명시)
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# OpenCV + 유틸
!pip install opencv-python numpy pandas


### YOLOv8n-Pose 불러오기 + TCN 연결 스켈레톤

In [ ]:
from ultralytics import YOLO
import torch
import torch.nn as nn

# 1) Pretrained YOLOv8n-Pose 로드
yolo_pose = YOLO('yolov8n-pose.pt')  # 기본 COCO 포즈

# 2) 프레임에서 포즈 뽑기 (예시: 이미지 한 장)
results = yolo_pose('sample.jpg')
# results[0].keypoints.xy  # (num_people, num_kpts, 2)

# 3) 포즈 시퀀스를 만들어 TCN으로 이벤트 분류
class SimpleTCN(nn.Module):
    def __init__(self, in_dim, hid_dim=128, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, hid_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hid_dim, hid_dim, kernel_size=3, padding=1)
        self.head = nn.Linear(hid_dim, num_classes)

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)     # (B, C, T)
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        h = h.transpose(1, 2)     # (B, T, H)
        logits = self.head(h)     # (B, T, num_classes)
        return logits


## 3. MoViNet-A0/A1 (TensorFlow)

### 필요 라이브러리
* tensorflow (2.15 근처 버전 or Colab 기본)

* tensorflow_hub

* opencv-python

In [ ]:
!pip install --upgrade pip

# TensorFlow + TF Hub
!pip install -U "tensorflow>=2.15.0" tensorflow-hub

# 영상처리
!pip install opencv-python numpy


### MoViNet-A0/A1 로딩 스켈레톤 (TF Hub)

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# MoViNet-A0 예시 (모델 URL은 예시용, 실제 TF Hub 페이지에서 확인해서 교체)
model_url = "https://tfhub.dev/tensorflow/movinet/a0/base/kinetics-600/classification/3"
movinet = hub.KerasLayer(model_url, trainable=True)

# 입력 샘플: (batch, frames, height, width, channels)
dummy = tf.random.uniform((1, 32, 172, 172, 3))
logits = movinet(dummy)
print(logits.shape)


## 4. Pose 기반 제스처 분류기 (Pose + MLP/TCN)

### 필요 라이브러리
* mediapipe

* torch



In [ ]:
!pip install --upgrade pip

# MediaPipe + 기본 유틸
!pip install mediapipe opencv-python

# Torch (Colab은 보통 torch 이미 설치되어 있으므로 버전 확인 후 필요 시 맞추기)
import torch, torchvision, torchaudio
print(torch.__version__)

# 그래도 명시적으로 깔고 싶으면 (CUDA 12.x용 최신 stable)
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

### MediaPipe Pose + MLP/TCN (PyTorch) 기반 분류

#### 제스쳐 기반 분류기

In [ ]:
import torch
import torch.nn as nn

class PoseGestureMLP(nn.Module):
    def __init__(self, in_dim, hid_dim=128, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.ReLU(),
            nn.Linear(hid_dim, hid_dim),
            nn.ReLU(),
            nn.Linear(hid_dim, num_classes)
        )

    def forward(self, x):
        # x: (B, C) 혹은 (B, T*C) 같은 평탄화된 포즈 피처
        return self.net(x)


#### 시퀸스 기반 TCN 분류

In [ ]:
class PoseGestureTCN(nn.Module):
    def __init__(self, in_dim, hid_dim=128, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, hid_dim, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hid_dim, hid_dim, kernel_size=3, padding=1)
        self.head = nn.Linear(hid_dim, num_classes)

    def forward(self, x):
        # x: (B, T, C)
        x = x.transpose(1, 2)             # (B, C, T)
        h = self.relu(self.conv1(x))
        h = self.relu(self.conv2(h))
        h = h.mean(dim=-1)                # Global avg over time → (B, H)
        logits = self.head(h)             # (B, num_classes)
        return logits


### YOLOv8n-Pose + MLP/TCN 기반 분류
* YOLOv8n-Pose 설치 필요
* MLP/TCL에 넣어 작동

In [ ]:
!pip install ultralytics opencv-python numpy pandas
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


#### 간단 TCL

In [ ]:
import torch
import torch.nn as nn

class SimpleTCN(nn.Module):
    def __init__(self, in_dim, hid_dim=128, num_classes=7, num_layers=3):
        super().__init__()
        layers = []
        dilation = 1
        for _ in range(num_layers):
            layers.append(
                nn.Conv1d(in_dim, hid_dim, kernel_size=3, padding=dilation, dilation=dilation)
            )
            layers.append(nn.ReLU())
            in_dim = hid_dim
            dilation *= 2
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(hid_dim, num_classes)

    def forward(self, x):
        # x: (B, T, C) -> Conv1d는 (B, C, T)
        x = x.transpose(1, 2)
        h = self.tcn(x)              # (B, H, T)
        h = h.transpose(1, 2)        # (B, T, H)
        logits = self.head(h)        # (B, T, num_classes)
        return logits


## 추가 유의사항
*   TensorFlow vs PyTorch 충돌

    *   같은 노트북에서 TF와 최신 torch를 같이 써도 보통 문제 없지만 드물게 CUDA 관련 라이브러리 충돌이 나면 → TF 쪽만 CPU로 쓰거나, 노트북을 분리해서 쓰는 게 안전.

*   ultralytics + torch 버전

    *   ultralytics는 일반적으로 최신 torch와 호환.

        만약 에러 나면:

        ```
        !pip install "torch==2.3.0" "torchvision==0.18.0" --index-url https://download.pytorch.org/whl/cu121
        !pip install --upgrade ultralytics

        ```
*   mediapipe
    *   Colab에서 !pip install mediapipe 최신 버전으로 쓰면 거의 문제 없음.

    *   OpenCV 버전이 너무 높아서 충돌 나는 케이스가 있었는데, 요즘은 거의 없고, 혹시나 하면:

        ```
        !pip install "opencv-python<4.9.0.0"

        ```


